# Qwen3-14B 목표 추적형 동화 QLoRA 노트북

이 버전은 이전 에러의 원인이었던 `look-behind requires fixed-width pattern` 문제를 제거했다.  
또한 선택지를 `1번=행동 / 2번=조사 / 3번=감정`처럼 프롬프트로 고정하지 않는다.

목표는 다음이다.

- Qwen3-14B를 QLoRA로 짧게 학습한다.
- 기존 `train.jsonl`, `val.jsonl`을 읽는다.
- 기존 `CHOICE` 템플릿 데이터는 기본적으로 학습에서 제외한다.
- 모델은 `최종 목표`와 `현재 중간 목표`를 보고 이야기가 산으로 가지 않게 이어 쓰는 능력을 배운다.
- 생성할 때 선택지는 모델이 장면 흐름에 맞게 알아서 판단해서 붙인다.

권장 GPU: A100 40GB 이상.  
T4에서는 Qwen3-14B QLoRA도 매우 빡셀 수 있다.


In [ ]:
# 설치
!pip -q install -U "transformers>=4.51.0" "accelerate>=0.34.0" "peft>=0.13.0" "bitsandbytes>=0.43.3" "datasets>=2.20.0" "trl>=0.10.1" sentencepiece einops


In [ ]:
# GPU 확인
import torch, os, platform
print("python:", platform.python_version())
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))
    props = torch.cuda.get_device_properties(0)
    print("vram GB:", round(props.total_memory / 1024**3, 2))


In [ ]:
# 파일 업로드: train.jsonl, val.jsonl 업로드
from google.colab import files
uploaded = files.upload()
print("uploaded files:", list(uploaded.keys()))


In [ ]:
# 설정
MODEL_NAME = "Qwen/Qwen3-14B"

TRAIN_PATH = "/content/train.jsonl"
VAL_PATH = "/content/val.jsonl"

# 30분 테스트용. 더 오래 돌릴 때만 올려라.
MAX_STEPS = 180
MAX_SEQ_LEN = 2048

# 기존 CHOICE 데이터는 템플릿이 강해서 기본 제외.
# 네가 올린 기존 선택지 데이터는 1/2/3 역할이 너무 고정되어 있어서 모델이 그 패턴을 외울 수 있다.
INCLUDE_OLD_CHOICE_ROWS = False

# 학습 데이터가 너무 많으면 빠른 테스트를 위해 일부만 사용.
MAX_TRAIN_EXAMPLES = 6000
MAX_VAL_EXAMPLES = 800

# LoRA 설정
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05

# 생성에서 선택지 형식 강제 여부. None이면 개수/방향을 모델이 알아서 판단.
DEFAULT_CHOICE_COUNT = None


In [ ]:
# JSONL 로드 + 안전한 텍스트 추출
import json, re, random, math, os
from pathlib import Path
from typing import List, Dict, Any, Optional

random.seed(42)


def read_jsonl(path: str) -> List[Dict[str, Any]]:
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for i, line in enumerate(f, 1):
            line = line.strip()
            if not line:
                continue
            try:
                rows.append(json.loads(line))
            except Exception as e:
                print(f"JSON parse skip line {i}: {e}")
    return rows


def clean_task_tail(text: str) -> str:
    text = str(text or "")
    text = text.replace("</TASK>", "")
    text = re.sub(r"<\/?(?:STORY|CONTINUATION|CHOICES|TASK|OUTPUT)>", "", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def extract_block(text: str, tag: str) -> str:
    m = re.search(rf"<{tag}>\s*(.*?)\s*</{tag}>", str(text or ""), re.S)
    return m.group(1).strip() if m else ""


def extract_story_state(prompt: str) -> Dict[str, str]:
    block = extract_block(prompt, "STORY_STATE")
    data = {}
    for line in block.splitlines():
        line = line.strip()
        if not line or ":" not in line:
            continue
        k, v = line.split(":", 1)
        data[k.strip()] = v.strip()
    return data


def normalize_space(text: str) -> str:
    text = str(text or "").replace("\r\n", "\n")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def split_sentences_ko(text: str) -> List[str]:
    """
    이전 버전의 에러 원인:
    (?<= ... | 다\. | 니다\. ) 처럼 길이가 다른 look-behind를 사용하면 Python re에서 실패한다.

    이 함수는 look-behind를 전혀 쓰지 않고 문장을 자른다.
    완벽한 문장분리기는 아니지만 동화 장면 청크 생성에는 충분히 안정적이다.
    """
    text = normalize_space(text)
    if not text:
        return []

    # 우선 명확한 문장부호 기준으로 자른다.
    parts = []
    buf = []
    end_chars = set(".!?。！？…")
    for ch in text:
        buf.append(ch)
        if ch in end_chars:
            s = "".join(buf).strip()
            if s:
                parts.append(s)
            buf = []

    rest = "".join(buf).strip()
    if rest:
        # 문장부호가 거의 없는 한국어 텍스트를 위해 종결어미 뒤 공백 기준으로 한 번 더 자른다.
        # look-behind 없이 캡처 그룹으로 처리한다.
        rough = re.split(r"((?:다|요|죠|니다|까요|어요|예요|했어요|였어요|있어요|없어요|해요)\s+)", rest)
        merged = []
        cur = ""
        for x in rough:
            cur += x
            if re.search(r"(?:다|요|죠|니다|까요|어요|예요|했어요|였어요|있어요|없어요|해요)\s+$", x):
                merged.append(cur.strip())
                cur = ""
        if cur.strip():
            merged.append(cur.strip())
        parts.extend(merged)

    # 너무 짧은 조각 제거
    out = []
    for p in parts:
        p = p.strip()
        if len(p) >= 2:
            out.append(p)
    return out if out else [text]


SCENE_MIN_CHARS = 220
SCENE_MAX_CHARS = 750


def split_into_scenes(story: str, min_chars: int = SCENE_MIN_CHARS, max_chars: int = SCENE_MAX_CHARS) -> List[str]:
    sents = split_sentences_ko(story)
    scenes, buf = [], ""
    for sent in sents:
        if not buf:
            buf = sent
        elif len(buf) + len(sent) + 1 <= max_chars:
            buf += " " + sent
        else:
            if len(buf) >= min_chars:
                scenes.append(buf.strip())
                buf = sent
            else:
                buf += " " + sent
    if buf.strip():
        if scenes and len(buf) < min_chars:
            scenes[-1] = (scenes[-1] + " " + buf).strip()
        else:
            scenes.append(buf.strip())
    return scenes


# 빠른 테스트
_test = "토끼는 숲으로 갔어요. 하지만 길을 잃었어요! 그래서 별빛 씨앗을 찾기로 했어요."
print(split_sentences_ko(_test))
print(split_into_scenes(_test, min_chars=10, max_chars=40))


In [ ]:
# 기존 JSONL을 목표 추적 학습 데이터로 변환
# 핵심: 선택지 방향을 프롬프트에서 고정하지 않는다.
# 기존 CHOICE rows는 기본 제외한다.

train_rows = read_jsonl(TRAIN_PATH)
val_rows = read_jsonl(VAL_PATH)
print("raw train:", len(train_rows), "raw val:", len(val_rows))


def make_final_goal(state: Dict[str, str], title: Optional[str], response: str) -> str:
    main = state.get("중심 인물") or "주인공"
    goal = state.get("목표") or "중요한 문제를 해결하기"
    mood = state.get("분위기") or "따뜻한 동화"
    lesson = state.get("교훈") or "작은 깨달음"
    title_txt = f"『{title}』에서 " if title else ""
    return f"{title_txt}{main}이(가) {goal} 위해 나아가고, 마지막에는 {lesson}을 자연스럽게 깨닫는 {mood}로 마무리된다."


def make_mid_goal(state: Dict[str, str], task: str, user_choice: str, response: str) -> str:
    # 중간 목표는 선택지 번호/역할을 고정하지 않고, 현재 장면이 다음에 향해야 할 방향만 짧게 준다.
    goal = state.get("목표") or "문제를 해결하기"
    conflict = state.get("현재 갈등") or "남아 있는 갈등"
    clue = state.get("중요한 물건/단서") or "중요한 단서"

    if user_choice:
        return f"사용자가 고른 흐름을 반영해 {goal}에 한 걸음 가까워진다. 선택: {user_choice}"
    if task == "STORY":
        return f"처음 장면에서 {conflict}과 {clue}를 자연스럽게 드러내며 {goal} 방향으로 출발한다."
    if task == "CONTINUE":
        return f"현재 갈등을 조금 진전시키되, 이야기가 {goal}에서 벗어나지 않게 이어진다."
    return f"다음 장면이 {goal}을 향하도록 자연스러운 갈림길을 만든다."


def build_user_prompt(final_goal: str, current_story: str, mid_goal: str, title: Optional[str] = None, task: str = "STORY") -> str:
    title_line = f"제목 힌트: {title}\n" if title else ""
    current_line = current_story.strip()
    if current_line:
        current_part = f"\n현재까지의 이야기:\n{current_line}\n"
    else:
        current_part = "\n현재까지의 이야기:\n아직 시작되지 않았다.\n"

    # 고정 선택지 형식 금지: 개수, 번호 역할, 행동/조사/감정 카테고리 강제 없음.
    return f"""너는 어린이를 위한 동화 작가다.
{title_line}
최종 목표:
{final_goal}
{current_part}
현재 중간 목표:
{mid_goal}

해야 할 일:
- 최종 목표와 현재 중간 목표에서 벗어나지 않게 다음 이야기를 쓴다.
- 갑자기 전혀 다른 사건, 장소, 인물을 많이 추가하지 않는다.
- 아이가 읽기 쉬운 따뜻한 문장으로 쓴다.
- 다음에 독자가 고를 수 있는 선택지가 필요하다고 판단되면, 이야기 흐름에 맞게 자연스럽게 덧붙인다.
- 선택지의 개수, 표현 방식, 방향은 미리 정해진 틀을 따르지 말고 현재 장면에 맞춰 스스로 정한다.
""".strip()


def row_to_example(row: Dict[str, Any]) -> Optional[Dict[str, str]]:
    task = str(row.get("task", "STORY")).upper()
    if task == "CHOICE" and not INCLUDE_OLD_CHOICE_ROWS:
        return None

    prompt = row.get("prompt", "")
    response = clean_task_tail(row.get("response", ""))
    if not response or len(response) < 30:
        return None

    # 기존 CHOICE 응답은 템플릿성이 강하므로 기본 제외.
    # 만약 INCLUDE_OLD_CHOICE_ROWS=True로 켠 경우에도 프롬프트는 고정 형식을 제거해서 재구성한다.
    state = extract_story_state(prompt)
    metadata = row.get("metadata", {}) or {}
    title = metadata.get("title")
    context = extract_block(prompt, "CONTEXT")
    user_choice = extract_block(prompt, "USER_CHOICE")

    final_goal = make_final_goal(state, title, response)
    mid_goal = make_mid_goal(state, task, user_choice, response)

    user_prompt = build_user_prompt(final_goal, context, mid_goal, title=title, task=task)

    return {
        "user": user_prompt,
        "assistant": response,
        "task": task,
        "title": title or "",
        "final_goal": final_goal,
        "mid_goal": mid_goal,
    }


def build_examples(rows: List[Dict[str, Any]], max_examples: Optional[int] = None) -> List[Dict[str, str]]:
    examples = []
    for r in rows:
        ex = row_to_example(r)
        if ex:
            examples.append(ex)

    # 추가: 긴 STORY/CONTINUE 응답을 장면 pair로 쪼개서 goal-controlled continuation 예시를 만든다.
    # 여기서도 선택지 형식은 강제하지 않는다.
    extra = []
    for r in rows:
        task = str(r.get("task", "")).upper()
        if task == "CHOICE":
            continue
        prompt = r.get("prompt", "")
        response = clean_task_tail(r.get("response", ""))
        if len(response) < 500:
            continue
        state = extract_story_state(prompt)
        metadata = r.get("metadata", {}) or {}
        title = metadata.get("title")
        final_goal = make_final_goal(state, title, response)
        scenes = split_into_scenes(response)
        if len(scenes) < 2:
            continue
        for i in range(len(scenes) - 1):
            current_story = " ".join(scenes[:i+1])[-1400:]
            next_scene = scenes[i+1]
            # 다음 장면 첫 부분을 중간 목표로 사용한다. 고정 카테고리가 아니라 extractive target이다.
            preview = next_scene[:180].strip()
            mid_goal = f"다음 장면은 앞 이야기와 이어지며 이런 방향으로 나아간다: {preview}"
            user_prompt = build_user_prompt(final_goal, current_story, mid_goal, title=title, task="CONTINUE")
            extra.append({
                "user": user_prompt,
                "assistant": next_scene,
                "task": "SCENE_CONTINUE",
                "title": title or "",
                "final_goal": final_goal,
                "mid_goal": mid_goal,
            })

    examples.extend(extra)
    random.shuffle(examples)
    if max_examples:
        examples = examples[:max_examples]
    return examples

train_examples = build_examples(train_rows, MAX_TRAIN_EXAMPLES)
val_examples = build_examples(val_rows, MAX_VAL_EXAMPLES)

print("train examples:", len(train_examples))
print("val examples:", len(val_examples))
from collections import Counter
print("train task counts:", Counter(e["task"] for e in train_examples))
print("val task counts:", Counter(e["task"] for e in val_examples))
print("\n--- sample user ---\n", train_examples[0]["user"][:1200])
print("\n--- sample assistant ---\n", train_examples[0]["assistant"][:800])


In [ ]:
# Qwen3 tokenizer 로드 및 chat template 적용
from transformers import AutoTokenizer

_tokenizer_kwargs = {"trust_remote_code": True}
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, **_tokenizer_kwargs)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def apply_qwen_chat_template(user: str, assistant: Optional[str] = None, add_generation_prompt: bool = False) -> str:
    messages = [{"role": "user", "content": user}]
    if assistant is not None:
        messages.append({"role": "assistant", "content": assistant})
    try:
        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=add_generation_prompt,
            enable_thinking=False,
        )
    except TypeError:
        # transformers/tokenizer 버전 차이 대응
        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=add_generation_prompt,
        )

train_texts = [{"text": apply_qwen_chat_template(e["user"], e["assistant"])} for e in train_examples]
val_texts = [{"text": apply_qwen_chat_template(e["user"], e["assistant"])} for e in val_examples]

print(train_texts[0]["text"][:1800])


In [ ]:
# Dataset 생성
from datasets import Dataset

train_ds = Dataset.from_list(train_texts)
val_ds = Dataset.from_list(val_texts)

print(train_ds)
print(val_ds)


In [ ]:
# 모델 로드: Qwen3-14B 4bit QLoRA
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
)

model.config.use_cache = False
model.gradient_checkpointing_enable()
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


In [ ]:
# 토크나이즈 + transformers Trainer 학습
# TRL 버전 차이 때문에 SFTTrainer 대신 기본 Trainer를 사용한다.
# prompt 부분까지 loss에 들어가지만, 짧은 30분 실험에서는 안정성이 더 중요하다.

from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

OUTPUT_DIR = "/content/qwen3_14b_goal_free_choice_lora"


def tokenize_batch(batch):
    tok = tokenizer(
        batch["text"],
        truncation=True,
        max_length=MAX_SEQ_LEN,
        padding=False,
    )
    tok["labels"] = [ids.copy() for ids in tok["input_ids"]]
    return tok

train_tok = train_ds.map(tokenize_batch, batched=True, remove_columns=train_ds.column_names)
val_tok = val_ds.map(tokenize_batch, batched=True, remove_columns=val_ds.column_names)

collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    max_steps=MAX_STEPS,
    warmup_ratio=0.03,
    logging_steps=10,
    eval_steps=60,
    save_steps=60,
    save_total_limit=2,
    bf16=True,
    fp16=False,
    optim="paged_adamw_8bit",
    lr_scheduler_type="cosine",
    report_to="none",
    gradient_checkpointing=True,
    remove_unused_columns=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    data_collator=collator,
)

trainer.train()


In [ ]:
# 어댑터 저장
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print("saved to", OUTPUT_DIR)


In [ ]:
# 생성 함수: 선택지 형식은 강제하지 않는다.
# 최종 목표와 현재 이야기만 줘도 되고, 중간 목표는 비워 두면 모델이 장면 흐름에 맞게 내부적으로 잡는다.

import torch


def build_generation_prompt(
    current_story: str,
    final_goal: str,
    mid_goal: Optional[str] = None,
    title: Optional[str] = None,
    choice_count: Optional[int] = DEFAULT_CHOICE_COUNT,
) -> str:
    title_line = f"제목 힌트: {title}\n" if title else ""
    if mid_goal is None or not str(mid_goal).strip():
        mid_goal_text = "현재 이야기와 최종 목표를 보고 다음 장면의 중간 목표를 스스로 정한다. 단, 목표 자체를 길게 설명하지 말고 이야기로 보여 준다."
    else:
        mid_goal_text = str(mid_goal).strip()

    if choice_count is None:
        choice_rule = "선택지가 필요하다고 판단되면, 장면 흐름에 맞게 자연스럽게 붙인다. 선택지의 개수와 표현 방식은 스스로 정한다."
    else:
        choice_rule = f"마지막에 다음 선택지 {choice_count}개를 붙인다. 단, 각 선택지의 방향은 미리 정해진 틀이 아니라 현재 장면과 목표에 맞게 스스로 정한다."

    return f"""너는 어린이 동화 앱의 작가다.
{title_line}
최종 목표:
{final_goal}

현재까지의 이야기:
{current_story}

현재 중간 목표:
{mid_goal_text}

작성 방식:
- 최종 목표에서 벗어나지 않게 다음 장면을 쓴다.
- 갑자기 다른 장르, 다른 사건, 너무 많은 새 인물로 빠지지 않는다.
- 어린이가 읽기 쉬운 따뜻한 문장으로 쓴다.
- {choice_rule}
""".strip()


@torch.inference_mode()
def generate_goal_story(
    current_story: str,
    final_goal: str,
    mid_goal: Optional[str] = None,
    title: Optional[str] = None,
    choice_count: Optional[int] = DEFAULT_CHOICE_COUNT,
    max_new_tokens: int = 700,
    temperature: float = 0.8,
    top_p: float = 0.9,
):
    user = build_generation_prompt(
        current_story=current_story,
        final_goal=final_goal,
        mid_goal=mid_goal,
        title=title,
        choice_count=choice_count,
    )
    text = apply_qwen_chat_template(user, assistant=None, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    out = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=temperature,
        top_p=top_p,
        repetition_penalty=1.08,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.pad_token_id,
    )
    gen = out[0][inputs["input_ids"].shape[-1]:]
    return tokenizer.decode(gen, skip_special_tokens=True).strip()


sample = generate_goal_story(
    title="별빛 씨앗을 찾는 토끼",
    final_goal="토끼 미루가 잃어버린 별빛 씨앗을 찾아 숲을 다시 환하게 만들고, 친구들과 힘을 합치는 법을 배운다.",
    current_story="토끼 미루는 어두워진 숲길에서 반짝임을 잃은 작은 씨앗을 발견했어요. 멀리서는 누군가 훌쩍이는 소리가 들렸고, 바람은 낡은 지도 한 장을 미루의 발밑으로 밀어 왔어요.",
    mid_goal="미루가 울음소리와 낡은 지도를 단서로 삼아 별빛 씨앗의 위치에 가까워진다.",
    choice_count=None,  # None이면 선택지 개수/형식 강제 없음
)
print(sample)


In [ ]:
# 어댑터 zip으로 다운로드
!cd /content && zip -r qwen3_14b_goal_free_choice_lora.zip qwen3_14b_goal_free_choice_lora >/dev/null
from google.colab import files
files.download("/content/qwen3_14b_goal_free_choice_lora.zip")


## 중요한 운영 팁

- 기존 `CHOICE` 행은 기본적으로 제외되어 있다. 네 기존 선택지 데이터는 너무 규칙적이라 모델이 선택지 패턴을 외울 수 있다.
- 선택지를 완전히 안 배우는 게 걱정되면 `INCLUDE_OLD_CHOICE_ROWS=True`로 바꿔도 되지만, 그러면 다시 템플릿 냄새가 날 수 있다.
- 앱에서 반드시 3개 선택지가 필요하면 생성 함수에서 `choice_count=3`만 넘겨라. 그래도 방향은 고정하지 않는다.
- 이야기 산으로 가는 문제는 `final_goal`과 `mid_goal`을 매 턴 넘기는 방식으로 줄이는 게 가장 안정적이다.
